In [1]:
%cd ../..

/home/eli/AnacondaProjects/epych


In [2]:
%env DASK_LOGGING__DISTRIBUTED=CRITICAL
%env OMPI_MCA_btl_sm_backing_directory=/mnt/data/tmp_storage
%env SPYTMPDIR=/mnt/data/tmp_storage
%env SPYLOGLEVEL=CRITICAL
%env SPYPARLOGLEVEL=CRITICAL

env: DASK_LOGGING__DISTRIBUTED=CRITICAL
env: OMPI_MCA_btl_sm_backing_directory=/mnt/data/tmp_storage
env: SPYTMPDIR=/mnt/data/tmp_storage
env: SPYLOGLEVEL=CRITICAL
env: SPYPARLOGLEVEL=CRITICAL


In [3]:
import collections
import glob
import functools
import logging
import math
import matplotlib.pyplot as plt
import numpy as np
import os
import pickle
import quantities as pq

import epych
from epych.statistics import alignment

[striatum:2922977] shmem: mmap: an error occurred while determining whether or not /tmp/ompi.striatum.1000/jf.0/2772303872/shared_mem_cuda_pool.striatum could be created.
[striatum:2922977] create_and_attach: unable to create shared memory BTL coordinating structure :: size 134217728 


In [4]:
%matplotlib inline

In [5]:
logging.basicConfig(level=logging.INFO)

In [6]:
CONDITIONS = ["go_gloexp", "go_seqctl", "lo_gloexp", "lonaive", "lo_rndctl", "igo_seqctl"]
PRETRIAL_SECONDS = 0.5
POSTTRIAL_SECONDS = 0.5

In [7]:
NWB_SUBJECTS = glob.glob('/mnt/data/000253/sub-*/')

In [8]:
PILOT_FILES = []

In [9]:
NUM_TRIALS = 0
ODDBALL_ONSET = 0.
ODDBALL_OFFSET = 0.

In [10]:
aligner = epych.statistics.alignment.AlignmentSummary.unpickle("/mnt/data/000253/visual_alignment")
AREA_COUNTER = collections.Counter()

In [11]:
def visual_align(signal):
    visual = signal.select_channels(["VIS" in loc for loc in signal.channels.location]).median_filter()
    area = alignment.location_prefix(None, visual)
    result = aligner.stats[area].align(AREA_COUNTER[area], visual)
    AREA_COUNTER[area] += 1
    return result

In [12]:
def samplings(s, subject_dir, cond):
    subject = subject_dir.split('/')[-2]
    sampling = epych.recording.Sampling.unpickle(subject_dir + "/" + cond).smap(visual_align)
    global ODDBALL_ONSET
    global ODDBALL_OFFSET
    global NUM_TRIALS
    ODDBALL_ONSET += sampling.trials['stim3_start'].sum()
    ODDBALL_OFFSET += sampling.trials['stim3_end'].sum()
    NUM_TRIALS += len(sampling.trials)
    logging.info("Loaded LFPs for %s in subject %s" % (cond, subject))
    yield sampling
    del sampling

In [13]:
def initialize_spectrum(key, signal):
    area = os.path.commonprefix([loc for loc in signal.channels.location])
    return epych.statistics.spectrum.PowerSpectrum(signal.df, signal.channels, signal.f0, taper="hann")

In [14]:
for cond in CONDITIONS:
    global AREA_COUNTER
    AREA_COUNTER = collections.Counter()
    for s, subject_dir in enumerate(sorted(NWB_SUBJECTS)):
        subject = subject_dir.split('/')[-2]
        if not os.path.exists(subject_dir + "/" + cond):
            continue
        summary = epych.statistic.Summary(alignment.location_prefix, initialize_spectrum)
        summary.calculate(samplings(s, subject_dir, cond))
        summary.pickle("/mnt/data/000253/%s/spectrum_%s" % (subject_dir, cond))
        del summary
        logging.info("Analyzed spectra from %s LFPs in subject %s" % (cond, subject))

INFO:root:Loaded LFPs for go_gloexp in subject sub-621890
INFO:root:Analyzed spectra from go_gloexp LFPs in subject sub-621890
INFO:root:Loaded LFPs for go_gloexp in subject sub-632485
INFO:root:Analyzed spectra from go_gloexp LFPs in subject sub-632485
INFO:root:Loaded LFPs for go_gloexp in subject sub-632487
INFO:root:Analyzed spectra from go_gloexp LFPs in subject sub-632487
INFO:root:Loaded LFPs for go_gloexp in subject sub-637542
INFO:root:Analyzed spectra from go_gloexp LFPs in subject sub-637542
INFO:root:Loaded LFPs for go_gloexp in subject sub-637908
INFO:root:Analyzed spectra from go_gloexp LFPs in subject sub-637908
INFO:root:Loaded LFPs for go_gloexp in subject sub-637909
INFO:root:Analyzed spectra from go_gloexp LFPs in subject sub-637909
INFO:root:Loaded LFPs for go_gloexp in subject sub-640507
INFO:root:Analyzed spectra from go_gloexp LFPs in subject sub-640507
INFO:root:Loaded LFPs for go_gloexp in subject sub-642507
INFO:root:Analyzed spectra from go_gloexp LFPs in sub